# Guía L1 — La factura de la ciudad · **Sesión 1: medir un mes**

**Cálculo Integral — Laboratorio computacional** · Universidad Nacional de Colombia, Sede Manizales

> Esta idea no salió de un libro. Salió de una ducha en Londres — y de un personaje que vive en la pared de tu casa y lleva décadas haciendo cálculo integral sin pedir crédito: **el contador de la luz**. La historia completa está en la guía impresa; léanla antes de tocar este cuaderno, porque este cuaderno es la mitad ejecutable de esa historia.

**Cómo usar este cuaderno en Colab:** primero guarden su propia copia (`Archivo → Guardar una copia en Drive`) y renómbrenla `L1_apellidos.ipynb`. Trabajen sobre la copia. Las celdas marcadas con ✏️ son de ustedes: hay que completarlas. Las celdas de texto marcadas con 🗣️ son para escribir conclusiones del equipo, con sus palabras.

---
## ⏸️ Antes de ejecutar nada: Momento 1 — Predecir

Está en la guía (P1, P2, P3), con la gráfica impresa de un hogar en enero de 2013. **En papel, a mano, 15 minutos.** Cuando terminen, tómenle foto: va en el informe. Si ya lo hicieron, sigan. Si no, este cuaderno puede esperar; la predicción no.

---
## 1 · Tres palabras nuevas antes del primer `import`

**CSV** (*comma-separated values*): el formato más humilde y universal para tablas. Un archivo de texto plano: cada línea es una fila, las columnas van separadas por comas. Sin colores, sin fórmulas. Es el idioma en que las instituciones del mundo publican lo que miden.

**API**: para lo que nos importa hoy, un mesero de datos. Una URL bien construida es un pedido; el servidor responde con la tabla. Hoy el archivo de Londres lo servimos congelado desde el repositorio del curso (así todos trabajamos con exactamente los mismos datos); en la sesión 2 sí pediremos las tarifas de la CHEC en vivo, por API, al Estado colombiano.

**Dataset**: una tabla con pretensiones — datos **más su documentación**: qué mide cada columna, en qué unidades, quién los recogió, cuándo. Una tabla sin unidades es un chisme sin fuente.

Nuestro dataset de hoy: el consumo eléctrico **diario** de miles de hogares reales de Londres, medido por contadores inteligentes entre 2011 y 2014 (proyecto *Low Carbon London*, UK Power Networks). Cada fila dice: este hogar, este día, consumió esta energía. Columnas originales: `LCLid` (código anónimo del hogar), `Date` (el día), `KWH` (la energía de ese día, en kilovatios-hora). **Esa columna es la altura del escalón** — ya verán.

In [ ]:
# Herramientas del día: pandas (tablas), numpy (cuentas) y matplotlib (gráficas).
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# El CSV congelado del curso. Esta URL la da el profesor / monitor.
RUTA = "https://raw.githubusercontent.com/cmorregof/labs-calculo-2026-2/main/datos/london_energy.csv.gz"   # el diario congelado (comprimido; pandas lo lee tal cual)

df = pd.read_csv(RUTA)
print("Columnas originales:", list(df.columns))

# Primer acto matemático del día: renombrar es declarar qué significa cada columna.
df.columns = ["hogar", "fecha", "kwh"]
df["fecha"] = pd.to_datetime(df["fecha"])
print("Datos cargados. Eso que acaba de pasar: 3 millones y medio de días medidos se volvieron una tabla en memoria.")

---
## 2 · Conocer el dataset (mirar antes de cruzar)

Antes de calcular **cualquier** cosa, respondemos con código cuatro preguntas: ¿cuántas filas? ¿cuántas columnas? ¿cuántos hogares distintos? ¿qué rango de fechas? A esto se le llama *conocer el dataset* y hacerlo primero es el equivalente a mirar antes de cruzar la calle.

In [ ]:
print("Filas y columnas:", df.shape)
df.head(10)   # las primeras 10 filas, para ver la cara de los datos

In [ ]:
# ✏️ Ejercicio 1 — Completen las líneas que faltan.
print("Hogares distintos:", df["hogar"].nunique())
print("Primera fecha:", df["fecha"].min().date())
# print("Última fecha:", ...)                          # ← su turno
# print("¿Cuántos días distintos hay en el estudio?:", ...)   # ← su turno

---
## 3 · El dominio de cada columna (los avisos en el piso)

En Londres, el piso te grita **LOOK LEFT** porque el peligro viene del lado que no esperas. **En un dataset nadie pinta avisos en el piso: los pintan ustedes.** Escribir el dominio de cada columna es decidir, *antes* de operar, qué valores son legales y desde dónde pueden venir los ilegales.

🗣️ **Completen esta tabla en esta misma celda (doble clic para editar):**

| Columna | Dominio matemático (notación de conjuntos) | ¿Discreto o continuo? | Valores ilegales posibles |
|---|---|---|---|
| `hogar` | | | |
| `fecha` | | | |
| `kwh` | | | |

Y una pregunta que en este curso importa más que nunca: ¿puede un contador registrar un **día negativo**? ¿Y un día **faltante** — un hueco en el calendario de un hogar — es un valor ilegal o es otra cosa peor?

In [ ]:
# La policía de dominios: comparamos los datos contra el dominio que declararon.
print("--- Resumen estadístico de kwh (un día de un hogar) ---")
print(df["kwh"].describe())
print()
print("Valores faltantes (NaN) por columna:")
print(df.isna().sum())
print()
print("¿Consumos negativos?:", (df["kwh"] < 0).sum())
print("¿Filas duplicadas (mismo hogar, mismo día)?:", df.duplicated(subset=["hogar", "fecha"]).sum())

In [ ]:
# Los huecos del calendario: el infractor favorito de este laboratorio.
# Contamos, para cada (hogar, mes), cuántos días de lectura hay.
df["mes"] = df["fecha"].dt.to_period("M")
dias_por_mes = df.groupby(["hogar", "mes"]).size()

print("Meses con calendario completo (28-31 días):", (dias_por_mes >= 28).sum())
print("Meses con huecos (menos de 25 días):      ", (dias_por_mes < 25).sum())
print()
print("Guarden este número: en la tarea puente, los huecos van a ser protagonistas.")

# ✏️ Ejercicio 2 — Juzgar y decidir.
# Si la policía encontró infractores (NaN, negativos, duplicados), decidan QUÉ HACER
# con cada tipo (eliminar, corregir, conservar con nota) y háganlo aquí.
# Regla de oro: toda decisión queda escrita en un comentario.
# df = df[ ... ]   # ← su decisión, si aplica
print("Filas después de la limpieza:", len(df))

---
## 4 · La escalera: un mes visto como función escalonada

Vamos con el hogar de la predicción: **`MAC002515`**, enero de 2013 — el mismo de la gráfica impresa. Un contador diario nos da 31 números; la matemática de la clase nos da la forma de mirarlos: una **función escalonada** sobre el intervalo $[0, 31]$ (los días del mes), con partición

$$P = \{t_0, t_1, \ldots, t_{31}\} = \{0, 1, 2, \ldots, 31\}$$

y regla $s(t) = s_k$ si $t_{k-1} < t < t_k$: durante el día $k$, el hogar consume a razón de $s_k$ kWh/día.

In [ ]:
HOGAR = "MAC002515"
mes_obj = pd.Period("2013-01")

escalera = (df[(df["hogar"] == HOGAR) & (df["mes"] == mes_obj)]
            .sort_values("fecha").reset_index(drop=True))
print("Días de lectura en el mes:", len(escalera), "(¿está completo el calendario?)")

s = escalera["kwh"].to_numpy()          # alturas de los escalones: s_1, ..., s_31
t = np.arange(len(s) + 1)               # partición: t_0=0, t_1=1, ..., t_31=31

plt.figure(figsize=(11, 4.5))
plt.step(t[:-1], s, where="post", linewidth=2, label="s(t): la escalera (lo que el contador midió)")
plt.plot(t[:-1] + 0.5, s, "o--", alpha=0.35, markersize=3, label="la línea que une puntos (¿mentira?)")
plt.xlabel("Día de enero de 2013")
plt.ylabel("Consumo del día (kWh/día)")
plt.title(f"El enero del hogar {HOGAR}: una función escalonada de verdad")
plt.legend()
plt.show()

# 🗣️ ¿Por qué la línea punteada es una mentira aquí? ¿Qué afirma sobre las 3 p.m. del día 8
#    que nadie midió? La escalera también es un supuesto — pero ¿cuál, exactamente?
# 🗣️ Miren el tramo plano de ~1.9 kWh/día en la segunda mitad del mes. El contador está
#    contando una historia sobre ese hogar. ¿Cuál creen que es? (Nadie lo sabe con certeza:
#    eso también es honestidad estadística.)

🗣️ **El modelo, por escrito (doble clic):** escriban aquí la partición $P$ de su mes, los primeros tres valores $s_1, s_2, s_3$ con sus unidades, y la regla completa de $s(t)$.

**Y la respuesta a P3:** ¿qué consumo tiene el hogar el 15 de enero a la medianoche exacta ($t = 15$, la frontera entre dos escalones)? Según la definición de la clase, en los puntos $t_k$ la función puede valer **lo que quiera** — y la energía del mes ni se entera. En Apostol eso no es un descuido: es un teorema chiquito, y ustedes lo acaban de ver en un dataset real.

---
## 5 · El producto, con unidades (la pregunta más importante del semestre)

Antes de sumar nada: ¿qué **es** $s_{15} \cdot (t_{15} - t_{14})$?

No sigan hasta poder decirlo con palabras y con unidades. Parece una pregunta boba y es donde el concepto de integral se gana o se pierde.

In [ ]:
k = 15   # el día 15 del mes
producto = s[k-1] * (t[k] - t[k-1])
print(f"s_{k} = {s[k-1]} kWh/día   (la altura del escalón: el ritmo de ese día)")
print(f"Δt_{k} = {t[k] - t[k-1]} día    (la base del escalón: lo que dura ese ritmo)")
print(f"s_{k} · Δt_{k} = {producto:.3f} kWh  ← energía: el aporte del día 15 al mes")

# 🗣️ Escríbanlo con palabras: "el día 15 el hogar consumió a razón de ___ kWh/día durante
#    ___ día, luego aportó ___ kWh al mes". (kWh/día) × (días) = kWh. La unidad del producto
#    es la unidad de lo que se acumula. Siempre.

---
## 6 · La integral, tres veces

La energía del mes es la suma de los aportes de los 31 días:

$$\int_{t_0}^{t_{31}} s(t)\,dt \;=\; \sum_{k=1}^{31} s_k\,(t_k - t_{k-1})$$

Ese símbolo se lo presentaron hoy en clase; aquí lo vamos a calcular de **tres formas** y las tres deben coincidir hasta el último decimal. Si no coinciden, algo está mal — y encontrarlo es matemáticas, no informática.

In [ ]:
# Forma (a): un ciclo que acumula término a término — la definición, en cámara lenta.
energia_a = 0.0
for k in range(1, len(t)):
    energia_a = energia_a + s[k-1] * (t[k] - t[k-1])

# Forma (b): la misma suma, vectorizada.
energia_b = np.sum(s * np.diff(t))

# Forma (c): el área bajo la escalera, rectángulo por rectángulo.
plt.figure(figsize=(11, 4.5))
plt.bar(t[:-1], s, width=1.0, align="edge", alpha=0.6, edgecolor="white")
plt.xlabel("Día de enero de 2013")
plt.ylabel("kWh/día")
plt.title(f"∫ s(t) dt = área de los rectángulos = {energia_b:.3f} kWh")
plt.show()

print(f"(a) ciclo:      {energia_a:.6f} kWh")
print(f"(b) np.sum:     {energia_b:.6f} kWh")
print(f"¿Coinciden?     {np.isclose(energia_a, energia_b)}")

# 🗣️ Confronten P1: ¿quién del equipo tuvo el mejor ojo con la gráfica impresa?
#    ¿Qué método usó para estimar? ¿Contó rectángulos sin saberlo?

---
## 7 · Agrupar sin traicionar

P2 preguntaba: si solo tuvieran la lectura **semanal** (la energía de cada semana repartida pareja en sus 7 días), ¿cambia la energía del mes? Constrúyanlo y compruébenlo.

In [ ]:
# ✏️ Ejercicio 3 — La escalera semanal.
escalera["semana"] = escalera["fecha"].dt.isocalendar().week.astype(int)
por_semana = escalera.groupby("semana").agg(total=("kwh", "sum"), dias=("kwh", "size"))
por_semana["ritmo"] = por_semana["total"] / por_semana["dias"]   # kWh/día, parejo en la semana

# Reconstruimos la escalonada gruesa día a día:
s_semanal = escalera["semana"].map(por_semana["ritmo"]).to_numpy()

energia_semanal = np.sum(s_semanal * np.diff(t))
print(f"Integral de la escalera diaria:  {energia_b:.6f} kWh")
print(f"Integral de la escalera semanal: {energia_semanal:.6f} kWh")

plt.figure(figsize=(11, 4.5))
plt.step(t[:-1], s, where="post", linewidth=1.5, alpha=0.6, label="partición fina (días)")
plt.step(t[:-1], s_semanal, where="post", linewidth=2.5, label="partición gruesa (semanas)")
plt.xlabel("Día de enero de 2013"); plt.ylabel("kWh/día")
plt.title("Dos escaleras, la misma energía")
plt.legend(); plt.show()

# 🗣️ La energía total es la MISMA y la gráfica es otra. ¿Qué preguntas sobre el hogar
#    sobreviven al agrupamiento y cuáles mueren? (El viaje de mitad de mes, por ejemplo,
#    ¿se sigue viendo?) Acaban de usar, sin permiso, la aditividad de la integral en
#    subintervalos. En la clase tiene nombre; aquí tiene consecuencias.

---
## 8 · Álgebra de escaleras

La clase demostró dos propiedades: $\int (s + u) = \int s + \int u$ (aditiva) y $\int c\,s = c \int s$ (homogénea). Vamos a *ejecutarlas* sobre hogares de verdad — y a dejar una apuesta sembrada para la sesión 2.

In [ ]:
# ✏️ Ejercicio 4 — La casa doble y la casa triplicada.
OTRO = "MAC003740"   # ← cambien por el hogar que quieran (que tenga el mes completo)

s2 = (df[(df["hogar"] == OTRO) & (df["mes"] == mes_obj)]
      .sort_values("fecha")["kwh"].to_numpy())
print("Días del segundo hogar:", len(s2), "(si no son 31, elijan otro y expliquen por qué)")

# Propiedad aditiva: la "casa doble" que consume lo de ambos hogares.
int_suma  = np.sum((s + s2) * np.diff(t))
suma_ints = energia_b + np.sum(s2 * np.diff(t))
print(f"∫(s+u) = {int_suma:.6f}   |   ∫s + ∫u = {suma_ints:.6f}   |   ¿iguales? {np.isclose(int_suma, suma_ints)}")

# Propiedad homogénea: el hogar que triplica su ritmo cada día.
int_triple = np.sum((3*s) * np.diff(t))
print(f"∫3s = {int_triple:.6f}   |   3∫s = {3*energia_b:.6f}   |   ¿iguales? {np.isclose(int_triple, 3*energia_b)}")

# 🗣️ APUESTA (va al informe): en la sesión 2 la energía se vuelve plata con la tarifa CHEC.
#    Una de estas dos propiedades se va a ROMPER cuando midamos pesos en vez de kWh.
#    ¿Cuál, y por qué? Escriban su apuesta ahora — sin borrarla después.

---
## 9 · El misterio de febrero

Doce meses, doce integrales. Grafiquemos el año completo de nuestro hogar — y miren con sospecha el mes más corto.

In [ ]:
anio = (df[(df["hogar"] == HOGAR) & (df["fecha"].dt.year == 2013)]
        .groupby("mes").agg(energia=("kwh", "sum"), dias=("kwh", "size")))
anio["promedio_diario"] = anio["energia"] / anio["dias"]
print(anio.round(2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(range(1, 13), anio["energia"])
ax[0].set_title("Energía del mes: ∫ del mes (kWh)")
ax[0].set_xlabel("Mes de 2013"); ax[0].set_xticks(range(1, 13))
ax[1].bar(range(1, 13), anio["promedio_diario"], color="darkorange")
ax[1].set_title("Promedio diario: ∫ / (días del mes)  (kWh/día)")
ax[1].set_xlabel("Mes de 2013"); ax[1].set_xticks(range(1, 13))
plt.tight_layout(); plt.show()

# 🗣️ En la gráfica izquierda, ¿febrero parece el tacaño del invierno? ¿Y en la derecha?
#    (Pista: cuenten los días.) Moraleja para el informe: comparar integrales de intervalos
#    distintos sin normalizar es comparar peras con meses. La cantidad justa se llama
#    VALOR PROMEDIO de la función:  (1/(b-a)) ∫ s(t) dt  — y acaba de resolver el misterio.

---
## 🗣️ Momento 3 — Explicar

Media página del equipo, aquí mismo: ¿qué predijeron (P1, P2, P3)? ¿Qué encontraron? ¿Cuál fue la discrepancia más interesante y a qué la atribuyen?

**Frase obligatoria del informe — complétenla con sus palabras y con las unidades correctas:** “La integral de una función escalonada mide…”

---
## 🤖 Protocolo de IA (recordatorio)

La IA es tutora y auditora, no autora. Intento propio antes de consultar; la IA no es la autoridad final; **todo uso va a la bitácora individual** (prompt → fragmento de respuesta → qué aceptaron / corrigieron / rechazaron). Para depurar: *"Este código da este error: [ambos]. No me des el código corregido; explícame el error y hazme una pregunta que me guíe."* Todo integrante debe poder explicar cualquier celda si el monitor la señala.

**Esta semana: la tarea puente** (está en la guía): descargar el dato crudo de Kaggle, verificar 9 integrales contra las de hoy, y juzgar la regla de los 25 días. **Próxima sesión:** las tarifas de la CHEC llegan por API, ustedes deciden qué estrato le toca a cada hogar londinense, y descubrimos por qué el orden en que se acumula la plata cambia la factura. Traigan **el recibo de energía de su casa**.